# Exercise 1.5.4.16 — implement `resample_advanced`

> Part of [Delta Drills](https://delta-drills.vercel.app) ARENA practice. When the test cell passes, your completion is reported back to your account automatically.

**Section:** `1.5.4 Superposition & SAEs`  
**Notebook:** `1.5.4_Toy_Models_of_Superposition_&_SAEs_exercises.ipynb`  
**Return to Delta Drills:** [https://delta-drills.vercel.app/?arena_exercise=1.5.4.16](https://delta-drills.vercel.app/?arena_exercise=1.5.4.16)


# [1.5.4] Toy Models of Superposition & Sparse Autoencoders (exercises)

> **ARENA [Streamlit Page](https://arena-chapter1-transformer-interp.streamlit.app/34_[1.5.4]_Toy_Models_of_Superposition_&_SAEs)**
>
> **Colab: [exercises](https://colab.research.google.com/github/callummcdougall/ARENA_3.0/blob/main/chapter1_transformer_interp/exercises/part54_toy_models_of_superposition_and_saes/1.5.4_Toy_Models_of_Superposition_&_SAEs_exercises.ipynb?t=20260329) | [solutions](https://colab.research.google.com/github/callummcdougall/ARENA_3.0/blob/main/chapter1_transformer_interp/exercises/part54_toy_models_of_superposition_and_saes/1.5.4_Toy_Models_of_Superposition_&_SAEs_solutions.ipynb?t=20260329)**

Please send any problems / bugs on the `#errata` channel in the [Slack group](https://join.slack.com/t/arena-uk/shared_invite/zt-3afdmdhye-Mdb3Sv~ss_V_mEaXEbkABA), and ask any questions on the dedicated channels for this chapter of material.

You can collapse each section so only the headers are visible, by clicking the arrow symbol on the left hand side of the markdown header cells.

Links to all other chapters: [(0) Fundamentals](https://arena-chapter0-fundamentals.streamlit.app/), [(1) Transformer Interpretability](https://arena-chapter1-transformer-interp.streamlit.app/), [(2) RL](https://arena-chapter2-rl.streamlit.app/).

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/headers/header-13-1.png" width="350">

# Introduction

Superposition is a crucially important concept for understanding how transformers work. A definition from Neel Nanda's glossary:

> Superposition is when a model represents more than n features in an $n$-dimensional activation space. That is, features still correspond to directions, but **the set of interpretable directions is larger than the number of dimensions**.

Why should we expect something like this to happen? In general, the world has way more features than the model has dimensions of freedom, and so we can't have a one-to-one mapping between features and values in our model. But the model has to represent these features somehow. Hence, it comes up with techniques for cramming multiple features into fewer dimensions (at the cost of adding noise and interference between features).

These exercises are specifically designed to dive deeply into the theory of superposition, focusing less on practical applications of SAEs with real language models. Although this theory underpins much of the current work on SAEs, engaging deeply with this chapter is not absolutely necessary for the more practical material in sections 1.3.3 and 1.4.2, which take you through using SAEs for concept discovery and circuit discovery respectively. If you want to speedrun the most useful parts of this section, we strongly recommend section 1️⃣ and 5️⃣, with a brief look at section 2️⃣ if you have time - you can probably skip the rest.

## Reading Material

* [200 COP in MI: Exploring Polysemanticity and Superposition](https://www.alignmentforum.org/posts/o6ptPu7arZrqRCxyz/200-cop-in-mi-exploring-polysemanticity-and-superposition), <b>15 mins</b>
    * Read the post, up to and including "Tips" (although some parts of it might make more sense after you've read the other things here).
* Neel Nanda's [Dynalist notes on superposition](https://dynalist.io/d/n2ZWtnoYHrU1s4vnFSAQ519J#z=3br1psLRIjQCOv2T4RN3V6F2), <b>10 mins</b>
    * These aren't long, you should skim through them, and also use them as a reference during these exercises.
* Anthropic's [Toy Models of Superposition](https://transformer-circuits.pub/2022/toy_model/index.html), <b>20 mins</b>
    * You should read up to & including the "Summary: A Hierarchy of Feature Properties" section.
    * The first few sections ("Key Results", "Definitions and Motivation", and "Empirical Phenomena" are particularly important).
    * We'll also be going through other parts of this paper as we work through the exercises.

## Content & Learning Objectives

### 1️⃣ Toy Models of Superposition: Superposition in a Nonprivileged Basis

In this section, you'll cover the basics of Anthropic's toy models of superposition. You'll learn through examples what superposition is, why it happens, and why it presents a problem for interpreting neural networks.

> ##### Learning Objectives
>
> - Understand the concept of superposition, and how it helps models represent a larger set of features
> - Understand the difference between superposition and polysemanticity
> - Learn how sparsity contributes to superposition
> - Understand the idea of the feature importance curve
> - Learn how feature correlation changes the nature and degree of superposition

### 2️⃣ Toy Models of Superposition: Superposition in a Privileged Basis

Superposition in a privileged basis is a bit different than a nonprivileged basis. Firstly, the model has a tendency to align more with the basis directions (even though it often needs some degree of misalignment in order to represent all features). Secondly, many privileged bases are privileged because they are performing some kind of computation on their inputs, rather than just storing representations, so it's important to understand how our model is able to perform this computation.

> ##### Learning Objectives
>
> - Understand the difference between neuron vs bottleneck superposition (or computational vs representational superposition)
> - Learn how models can perform computation in superposition, via the example `f(x) = abs(x)`

### 3️⃣ Feature Geometry

The section takes a slightly deeper dive into the geometry of superposition. It's not as essential as any exercises from the previous three sections, but should still be of interest and worth doing if you have time, or just want to dive more deeply into superposition. We cover **dimensionality** (a measure of how much capacity the model is affording to a single feature), and relate this metric to the forming of increasingly complex geometric structures.

> ##### Learning Objectives
>
> - Learn about *dimensionality*, which essentially measures what fraction of a dimension is allocated to a specific feature
> - Understand the geometric intuitions behind superposition, and how they relate to the more general ideas of superposition in larger models

### 4️⃣ Superposition & Deep Double Descent

Deep Double Descent is a long observed phenomena in deep learning, wherein the model's performance on a task first improves, then plateaus, then starts to improve again with further increases in model size, data size, or training time. The plateau is in line with the predictions of classical statistics (i.e. model overfitting), but the further decrease in loss isn't. Anthropic's paper on deep double descent connects this phenomenon to the idea of superposition, essentially arguing that the two scaling phases represent a memorizing solution and a generalizing solution respectively, with the first characterized by representing *datapoints* in superposition, the second by representing *features* in superposition.

This section guides you through a replication of this paper's key results. Relative to the other material here, this is quite unguided, so we recommend you treat it more as an optional extension than an exercise on the core pathway for today.

> ##### Learning Objectives
> 
> - Understand and characterise the deep double descent phenomena
> - Relate the different phases of double descent to the idea of superposition
> - Practice replicating results from a paper in a more unguided way

### 5️⃣ Sparse Autoencoders in Toy Models

In this last section, you'll learn about sparse autoencoders, and how they might help us resolve problems of superposition. You'll train a sparse autoencoder on the toy model setup from earlier sections, and you'll also implement techniques like neuron resampling and different architectures (e.g. the Gated architecture from [DeepMind's paper](https://deepmind.google/research/publications/88147/)).

> ##### Learning Objectives
>
> - Learn about sparse autoencoders, and how they might be used to disentangle features represented in superposition
> - Train your own SAEs on the toy models from earlier sections, and visualise the feature reconstruction process
> - Understand important SAE training strategies (e.g. resampling) and architecture variants (e.g. Gated, Jump ReLU)

### ☆ Bonus

We end with a section of suggested bonus material & paper replications, like usual.

## Questions

Here are a set of questions (with some brief answers provided) which you should be able to answer for yourself after reading the above material. Search for them on Neel's Dynalist notes if you didn't come across them during your reading.

What is a **privileged basis**? Why should we expect neuron activations to be privileged by default? Why *shouldn't* we expect the residual stream to be privileged?

<details>
<summary>Answer</summary>

A privileged basis is one where the **standard basis directions are meaningful** (due to the structure of computation being done on that basis). This doesn't necessarily mean that the basis is interpretable.

**Neurons**

Neuron activations are privileged because of the **elementwise nonlinear function that gets applied**. ReLU is easily described in the standard basis, e.g. in 2D: 

$$\begin{bmatrix} x \\ y \end{bmatrix} \to \begin{bmatrix} \max(x, 0) \\ \max(y, 0) \end{bmatrix}$$

but if you redefine a basis $x' = (x+y)/\sqrt{2}$, $y' = (x-y)/\sqrt{2}$, then describing ReLU in this new basis becomes really messy. More importantly, we now get interference between the components $x'$ and $y'$, i.e. the ReLU is no longer acting on them independently.

$$\begin{bmatrix} x' \\ y' \end{bmatrix} \to \frac{1}{\sqrt{2}} \begin{bmatrix} \max(x, 0) + \max(y, 0) \\ \max(x, 0) - \max(y, 0) \end{bmatrix} = \frac{1}{2} \begin{bmatrix} \max(x'+y', 0) + \max(x'-y', 0) \\ \max(x'+y', 0) - \max(x'-y', 0) \end{bmatrix}$$

**Residual stream**

The residual stream is not privileged because anything that reads from it and writes to it uses a linear map. As a thought experiment, if we changed all the writing matrices (i.e. $W_{out}$ in the MLP layers and $W_O$ in the attention layers) to $W \to W R$, and all the reading matrices (i.e. $W_{in}$ in the MLP layers and $W_Q$, $W_K$, $W_V$ in the attention layers) to $W \to W R^{-1}$ where $R$ is some arbitrary rotation matrix, then the model's computation would be unchanged. Since the matrix $R$ is arbitrary, it can change the basis in any way it wants, so that basis can't be privileged.

To put this another way - if you claimed "I think the 47th element of the residual stream encoded some special information e.g. the plurality of the noun at that sequence position", I could call bullshit on your claim, because this thought experiment shows that any basis direction could just as easily be rotated & distributed as a linear combination of several different basis directions without fundamentally changing the computation done by the transformer. The same does not apply to neurons, because a rotation / change of basis would change the nature of computation done on them.

**Summary**

**Something is a privileged basis if it is not rotation-independent**, i.e. the nature of computation done on it means that the **basis directions have some special significance.**

Common misconception: privileged basis is equivalent to interpretable basis. This is **NOT true** (although it is the case that a basis must be privileged if the individual basis directions have some interpretable meaning; this is necessary but not sufficient).

</details>

What is the difference between **superposition** and **polysemanticity**?

<details>
<summary>Answer</summary>

Polysemanticity happens when one neuron corresponds to multiple features (see [here](https://distill.pub/2020/circuits/zoom-in/#:~:text=lot%20of%20effort.-,Polysemantic%20Neurons,-This%20essay%20may) for more discussion & examples). If we only had polysemanticity, this wouldn't really be a problem for us (there might exist a basis for features s.t. each basis vector corresponds to a single feature).

Superposition is when there are **more features than dimensions**. So it implies polysemanticity (because we must have dimensions representing more than one feature), but the converse is not true.

</details>


What are the **importance** and **sparsity** of features? Do you expect more or less polysemantic neurons if sparsity is larger?

<details>
<summary>Answer</summary>

**Importance** = how useful is this feature for achieving lower loss?

**Sparsity** = how frequently is it in the input data?

If sparsity is larger, then we expect more polysemantic neurons. This is because a single neuron can afford to represent several different sparse features (usually it'll only be representing one of them at any given time, so there won't be interference).
</details>

How would you define a **feature**?

<details>
<summary>Answer</summary>

There's no single correct answer to this. Many of the definitions are unfortunately circular (e.g. "a feature is a thing which could be represented by a neuron"). A few possible definitions are this one from Neel's [Dynalist notes](https://dynalist.io/d/n2ZWtnoYHrU1s4vnFSAQ519J#q=feature):

> A feature is a property of an input to the model, or some subset of that input (eg a token in the prompt given to a language model, or a patch of an image).

or this similar one from Chris Olah's [Distill circuits Thread](https://distill.pub/2020/circuits/zoom-in/):

> A feature is a a scalar function of the input. In this essay, neural network features are directions, and often simply individual neurons. We claim such features in neural networks are typically meaningful features which can be rigorously studied. A **meaningful feature** is one that genuinely responds to an articulable property of the input, such as the presence of a curve or a floppy ear.
</details>

## Setup (don't read, just run)

In [ ]:
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

chapter = "chapter1_transformer_interp"
repo = "ARENA_3.0"
branch = "main"

# Install dependencies
try:
    import transformer_lens
except:
    %pip install einops datasets jaxtyping "sae-lens>=4.0.0,<5.0.0" tabulate eindex-callum transformer_lens==2.17.0
    %pip install --force-reinstall numpy pandas

# Get root directory, handling 3 different cases: (1) Colab, (2) notebook not in ARENA repo, (3) notebook in ARENA repo
root = (
    "/content"
    if IN_COLAB
    else "/root"
    if repo not in os.getcwd()
    else str(next(p for p in Path.cwd().parents if p.name == repo))
)

if Path(root).exists() and not Path(f"{root}/{chapter}").exists():
    if not IN_COLAB:
        !sudo apt-get install unzip
        %pip install jupyter ipython --upgrade

    if not os.path.exists(f"{root}/{chapter}"):
        !wget -P {root} https://github.com/callummcdougall/ARENA_3.0/archive/refs/heads/{branch}.zip
        !unzip {root}/{branch}.zip '{repo}-{branch}/{chapter}/exercises/*' -d {root}
        !mv {root}/{repo}-{branch}/{chapter} {root}/{chapter}
        !rm {root}/{branch}.zip
        !rmdir {root}/{repo}-{branch}


if f"{root}/{chapter}/exercises" not in sys.path:
    sys.path.append(f"{root}/{chapter}/exercises")

os.chdir(f"{root}/{chapter}/exercises")

In [ ]:
import sys
from collections import defaultdict
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Callable, Literal

import einops
import numpy as np
import pandas as pd
import plotly.express as px
import torch as t
from IPython.display import HTML, display
from jaxtyping import Float
from torch import Tensor, nn
from torch.distributions.categorical import Categorical
from torch.nn import functional as F
from tqdm.auto import tqdm

device = t.device("mps" if t.backends.mps.is_available() else "cuda" if t.cuda.is_available() else "cpu")

# Make sure exercises are in the path
chapter = "chapter1_transformer_interp"
section = "part54_toy_models_of_superposition_and_saes"
root_dir = next(p for p in Path.cwd().parents if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section

import part54_toy_models_of_superposition_and_saes.tests as tests
import part54_toy_models_of_superposition_and_saes.utils as utils
from plotly_utils import imshow, line

MAIN = __name__ == "__main__"

# 1️⃣ TMS: Superposition in a Nonprivileged Basis

> ##### Learning Objectives
>
> - Understand the concept of superposition, and how it helps models represent a larger set of features
> - Understand the difference between superposition and polysemanticity
> - Learn how sparsity contributes to superposition
> - Understand the idea of the feature importance curve
> - Learn how feature correlation changes the nature and degree of superposition

## Toy Model setup

In this section, we'll be examining & running experiments on the toy model studied in [Anthropic's paper](https://transformer-circuits.pub/2022/toy_model/index.html).

You can follow along with the paper from the [Demonstrating Superposition](https://transformer-circuits.pub/2022/toy_model/index.html#demonstrating) section onwards; it will approximately follow the order of the sections in this notebook.

This paper presented a very rudimentary model for **bottleneck superposition** - when you try and represent more than $n$ features in a vector space of dimension $n$. The model is as follows:

* We take a 5-dimensional input $x$
* We map it down into 2D space
* We map it back up into 5D space (using the transpose of the first matrix)
* We add a bias and ReLU

$$
\begin{aligned}
h &= W x \\
x' &= \operatorname{ReLU}(W^T h + b)
\end{aligned}
$$

### What's the motivation for this setup?

The input $x$ represents our five features (they're uniformly sampled between 0 and 1).

Each feature can have **importance** and **sparsity**. Recall our earlier definitions:

* **Importance** = how useful is this feature for achieving lower loss?
* **Sparsity** = how frequently is it in the input data?

This is realised in our toy model as follows:

* **Importance** = the coefficient on the weighted mean squared error between the input and output, which we use for training the model
    * In other words, our loss function is $L = \sum_x \sum_i I_i (x_i - x_i^\prime)^2$, where $I_i$ is the importance of feature $i$.
* **Sparsity** = the probability of the corresponding element in $x$ being zero
    * In other words, this affects the way our training data is generated (see the method `generate_batch` in the `Module` class below)
    * We often refer to **feature probability** (1 minus sparsity) rather than sparsity

The justification for using $W^T W$ is as follows: we can think of $W$ (which is a matrix of shape `(2, 5)`) as a grid of "overlap values" between the features and bottleneck dimensions. The values of the 5x5 matrix $W^T W$ are the dot products between the 2D representations of each pair of features. To make this intuition clearer, imagine each of the columns of $W$ were unit vectors, then $W^T W$ would be a matrix of cosine similarities between the features (with diagonal elements equal to 1, because the similarity of a feature with itself is 1). To see this for yourself:

In [ ]:
t.manual_seed(2)

W = t.randn(2, 5)
W_normed = W / W.norm(dim=0, keepdim=True)

imshow(
    W_normed.T @ W_normed,
    title="Cosine similarities of each pair of 2D feature embeddings",
    width=600,
)

To put it another way - if the columns of $W$ were orthogonal, then $W^T W$ would be the identity. This can't actually be the case because $W$ is a 2x5 matrix, but its columns can be "nearly orthgonal" in the sense of having pairwise cosine similarities close to 0.

<details>
<summary>Question - can you prove that <code>W.T @ W</code> can't be the identity when <code>W</code> has more columns than rows (or alternatively, when the hidden dimension is strictly smaller than the input dimension)?</summary>

Proof #1: the rank of a matrix product $AB$ is upper-bounded by the maximum of the two factors $A$ and $B$. In the case of $W^T W$, both matrices have rank at most 2, so the product has rank at most 2.

Proof #2: for any vector $x$, $W^T W x = W^T (Wx)$ is in the span of the columns of $W^T$, which is vector space with rank 2.

</details>

Another nice thing about using two bottleneck dimensions is that we get to visualise our output! We've got a few helper functions for this purpose.

In [ ]:
utils.plot_features_in_2d(
    W_normed.unsqueeze(0),  # shape [instances=1 d_hidden=2 features=5]
)

Compare this plot to the `imshow` plot above, and make sure you understand what's going on here (and how the two plots relate to each other). A lot of the subsequent exercises run with this idea of a geometric interpretation of the model's features and bottleneck dimensions.

<details>
<summary>Help - I'm confused about how these plots work.</summary>

As mentioned, you can view $W$ as being a set of five 2D vectors, one for each of our five features. The heatmap shows us the cosine similarities between each pair of these vectors, and the second plot shows us these five vectors in 2D space.

In the example above, we can see two pairs of vectors (the 1st & 2nd, and the 0th & 4th) have very high cosine similarity. This is reflected in the 2D plot, where these features are very close to each other (the 0th feature is the darkest color, the 4th feature is the lightest).

</details>

### Defining our model

Below is some code for your model (with most methods not filled out yet). It should be familiar to you if you've already built simple neural networks earlier in this course.

Some notes on the initialization method, which is filled out for you:

#### Weights & instances

The `Config` class has an `n_inst` class. This is so we can optimize multiple models at once in a single training loop (this'll be useful later on). You should treat this as basically like a batch dimension for your weights: each of your weights/biases will actually be `n_inst` separate weights/biases stacked along the zeroth dimension, and each of these will be trained independently, on different data, in parallel (using the same optimizer).

We initialize weights `W` and `b_final`, which correspond to $W$ and $b$ in the Anthropic paper.

#### Sparsity & Importance

The `feature_probability` argument tells us the probability that any given feature will be active. We have the relation  `feature_probability = 1 - sparsity`. We'll often be dealing with very small feature probabilities $p = 1 - S \approx 0$, i.e. sparsities close to 1. The feature probability is used to generate our training data; the importance is used in our loss function (see later for both of these). The default is `feature_probability = 0.01`, i.e. each feaure is present with probability 1%.

The `importance` argument is used when calculating loss (see later exercise). The default is `importance = None` which results in uniform importance.

In the `__init__` method, we have code to broadcast `feature_probability` and `importance`, so that by the end they both always have shape `(n_inst, n_features)`.

## Connect to Delta Drills

Paste your Delta Drills auth token below so this exercise can report its completion back to your account.
You can copy the token from your Delta Drills account page.


In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_EXERCISE_ID = "1.5.4.16"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"


### Prior-exercise solutions (auto-imported)

These were imported from ARENA's reference `solutions.py` so you can jump straight into this exercise without having implemented every predecessor. Re-implement them yourself if you'd rather build top-to-bottom.


In [ ]:
from part54_toy_models_of_superposition_and_saes.solutions import linear_lr, generate_batch, calculate_loss, generate_correlated_features, NeuronModel, NeuronModel, NeuronComputationModel, NeuronComputationModel, NeuronComputationModel, compute_dimensionality, __init__, W_dec_normalized, generate_batch, forward, resample_simple


### Exercise - implement `resample_advanced`


> ```yaml
> Difficulty: 🔴🔴🔴🔴🔴
> Importance: 🔵🔵⚪⚪⚪
> 
> You should spend up to 20-40 minutes on this exercise, if you choose to do it.
> ```

This section can be considered optional if you've already implemented the simpler version of `resample` above. However, if you're interested in a version of it which hues close to [Anthropic's methodology](https://transformer-circuits.pub/2023/monosemantic-features/index.html#appendix-autoencoder-resampling), then you might still be interested in this exercise.

The main difference we'll make is in how the resampled values are chosen. Rather than just drawing them randomly from a distribution and normalizing them, we'll be **sampling them with replacement from a set of input activations $h$, with sampling probabilities weighted by the squared $L_2$ loss of the autoencoder on each input**. Intuitively, this will make it more likely that our resampled neurons will represent feature directions that the autoencoder is currently doing a bad job of representing.

The new resampling algorithm looks like the following - for each instance we:

* Generate a batch of hidden data `h` from your SAE and compute its squared reconstruction loss `l2_squared`. It should have shape `(batch_size, n_inst)`. If the L2 loss for this instance `l2_squared[:, inst]` is zero everywhere, we can skip this instance.
* Find the dead latents for this instance (i.e. the instances `inst` and latent indices `d` where `frac_active_in_window[:, inst, d]` are all zero).
* For each of these, do the following:
    * Randomly sample a vector `v = h[x, inst, :]`, where `0 <= x < batch_size` is chosen according to the distribution with probabilities proportional to `l2_squared[:, inst]`.
    * Set the decoder weights `W_dec[inst, d, :]` to this new vector `v`, normalized.
    * Set the encoder weights `W_enc[inst, :, d]` to this new vector `v`, scaled to have norm `resample_scale * avg_W_enc_alive_norm` (where the term `avg_W_enc_alive_norm` is the mean norm of the encoder weights of alive neurons for that particular instance).
    * Set the encoder biases `b_enc[inst, d]` to zero.

So we really have just 2 changes: the added use of `avg_W_enc_alive_norm` for the encoder weights, and the sampling from the L2-based distribution to get our vectors `v`. Because this function can get a bit messy, we recommend you iterate through the instances rather than trying to resample them all at once.

For the sampling, we recommend that you use `torch.distributions.categorical.Categorical` to define a probability distribution, which can then be sampled from using the `sample` method. We've included an example of how to use this function below.

<details>
<summary>Example of using <code>Categorical</code>.</summary>

```python
from torch.distributions.categorical import Categorical

# Define a prob distn over (0, 1, 2, 3, 4) with probs proportional to (4, 3, 2, 1, 0)
values = t.arange(5).flip(0)
probs = values.float() / values.sum()
distribution = Categorical(probs = probs)

# Sample a single value from it
distribution.sample() # tensor(1)

# Sample multiple values with replacement (values will mostly be in the lower end of the range)
distribution.sample((10,)) # tensor([1, 1, 3, 0, 0, 1, 0, 3, 2, 2])
```

When you're sampling multiple times, make sure to pass a 1D tensor rather than a scalar.

</details>

Once you've implemented this resampling method, run the tests:

In [ ]:
# Go back up and edit your `ToySAE.resample_advanced` method, then run the test below

tests.test_resample_advanced(ToySAE)

<details><summary>Solution</summary>

```python
@t.no_grad()
def resample_advanced(
    self: ToySAE,
    frac_active_in_window: Float[Tensor, "window inst d_sae"],
    resample_scale: float,
    batch_size: int,
) -> None:
    """
    Resamples latents that have been dead for 'dead_feature_window' steps, according to `frac_active`.

    Resampling method is:
        - Compute the L2 reconstruction loss produced from the hidden state vectors `h`
        - Randomly choose values of `h` with probability proportional to their reconstruction loss
        - Set new values of W_dec & W_enc to be these centered & normalized vecs, at each dead neuron
        - Set b_enc to be zero, at each dead neuron

    Returns colors and titles (useful for creating the animation: resampled neurons appear in red).
    """
    h = self.generate_batch(batch_size)
    l2_loss = self.forward(h)[0]["L_reconstruction"]

    for instance in range(self.cfg.n_inst):
        # Find the dead latents in this instance. If all latents are alive, continue
        is_dead = (frac_active_in_window[:, instance] < 1e-8).all(dim=0)
        dead_latents = t.nonzero(is_dead).squeeze(-1)
        n_dead = dead_latents.numel()
        if n_dead == 0:
            continue  # If we have no dead features, then we don't need to resample

        # Compute L2 loss for each element in the batch
        l2_loss_instance = l2_loss[:, instance]  # [batch_size]
        if l2_loss_instance.max() < 1e-6:
            continue  # If we have zero reconstruction loss, we don't need to resample

        # Draw `d_sae` samples from [0, 1, ..., batch_size-1], with probabilities proportional to
        # the values of l2_loss
        distn = Categorical(probs=l2_loss_instance.pow(2) / l2_loss_instance.pow(2).sum())
        replacement_indices = distn.sample((n_dead,))  # type: ignore

        # Index into the batch of hidden activations to get our replacement values
        replacement_values = (h - self.b_dec)[replacement_indices, instance]  # [n_dead d_in]
        replacement_values_normalized = replacement_values / (
            replacement_values.norm(dim=-1, keepdim=True) + self.cfg.weight_normalize_eps
        )

        # Get the norm of alive neurons (or 1.0 if there are no alive neurons)
        W_enc_norm_alive_mean = self.W_enc[instance, :, ~is_dead].norm(dim=0).mean().item() if (~is_dead).any() else 1.0

        # Lastly, set the new weights & biases (W_dec is normalized, W_enc needs specific scaling,
        # b_enc is zero)
        self.W_dec.data[instance, dead_latents, :] = replacement_values_normalized
        self.W_enc.data[instance, :, dead_latents] = (
            replacement_values_normalized.T * W_enc_norm_alive_mean * resample_scale
        )
        self.b_enc.data[instance, dead_latents] = 0.0


ToySAE.resample_advanced = resample_advanced
```
</details>

After passing the tests, you can try training & visualizing your SAE again. You might not spot a lot of improvement with this resampling method in 2 dimensions, but for much higher-dimensional spaces it becomes highly beneficial to resample neurons in a more targeted way.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

def _dd_report_complete():
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    try:
        body = _dd_json.dumps({
            'exercise_id': DD_EXERCISE_ID,
            'passed': True,
        }).encode('utf-8')
        req = _dd_req.Request(
            f'{DD_BACKEND_URL}/api/arena/complete',
            data=body,
            headers={
                'Content-Type': 'application/json',
                'Authorization': f'Bearer {DD_TOKEN}',
            },
            method='POST',
        )
        with _dd_req.urlopen(req, timeout=3) as r:
            r.read()
        print(f'[Delta Drills] reported completion of {DD_EXERCISE_ID}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

# Wrap tests.test_resample_advanced so a passing test fires the beacon.
try:
    _dd_orig = tests.test_resample_advanced
    def _dd_wrapped(*args, **kwargs):
        result = _dd_orig(*args, **kwargs)
        _dd_report_complete()
        return result
    tests.test_resample_advanced = _dd_wrapped
except AttributeError:
    print('[Delta Drills] no matching test function — call _dd_report_complete() manually when done.')
